###### Creating the Workspace & Project

In [1]:
import pandas as pd
import warnings
from evidently.ui.workspace import CloudWorkspace
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
import getpass
from evidently import Dataset, DataDefinition, BinaryClassification
from evidently.presets import DataDriftPreset, DataSummaryPreset, ClassificationPreset 
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot
from evidently.presets import ClassificationPreset

warnings.filterwarnings("ignore", category=RuntimeWarning)

###### Data Loading & Preparation

In [2]:
# project data
train_df = pd.read_csv('../data/processed/train_processed.csv')
test_df = pd.read_csv('../data/processed/test_processed.csv')

for df in [train_df, test_df]:
    if 'is_default' not in df.columns:
        df['is_default'] = df['loan_status'].apply(lambda x: 1 if 'Default' in str(x) else 0)
    
    # Logic: High Inflation + High DTI = Higher chance of predicted default
    if 'prediction' not in df.columns:
        # Simple threshold simulation for the report logic
        df['prediction'] = (df['dti'] * df['inflation'] > 200).astype(int)

print(f"Data Loaded: Train({train_df.shape}) | Test({test_df.shape})")

C:\Users\JAMES TECH\AppData\Local\Temp\ipykernel_7644\2677304094.py:2: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv('../data/processed/train_processed.csv')
C:\Users\JAMES TECH\AppData\Local\Temp\ipykernel_7644\2677304094.py:3: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv('../data/processed/test_processed.csv')


Data Loaded: Train((80000, 157)) | Test((20000, 157))


###### Data Definition

In [3]:
# Define the Schema (Numerical & Categorical ONLY)
schema = DataDefinition(
    numerical_columns=['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'inflation', 'macro_stress_index'],
    categorical_columns=['term', 'grade']
)

# Wrap in Evidently Datasets
eval_data_1 = Dataset.from_pandas(pd.DataFrame(test_df), data_definition=schema)
eval_data_2 = Dataset.from_pandas(pd.DataFrame(train_df), data_definition=schema)

In [4]:
eval_data_1

In [5]:
eval_data_2

###### Creating a project

In [16]:
project = ws.create_project("MASTS-Stress-Testing-Monitoring", org_id="019a39aa-0a3e-7d81-b103-2b3d56792567")
project.description = "Individual project for monitoring Nigerian Inflation Stress Tests."
project.save()

#project = ws.get_project("PROJECT_ID")
#print(project)

###### Generating Report and Drifts

In [ ]:
sk_prod.019d29e4-638b-73fb-9311-9991cb05d7eb.Ezatrs6MHk6muon1SA6Yw3mTnT76vcMwSPoraBJimxarBr3YEjNatn7r0eSRFyAi3ySeT5QzXTL70anC3lgEyKo_BaOdEJCDtp2Es7CrOOwJ63mH7FPxhS6Mexjnqyr2

In [6]:
# RE-CONNECT
api_token = getpass.getpass("Enter your Evidently Cloud API Token: ") 
ws = CloudWorkspace(token=api_token, url="https://app.evidently.cloud")

# DIRECT PROJECT RECOVERY (Using specific ID)
PROJECT_ID = "019d2e74-a7e4-7383-84ac-15385631afc4"

try:
    project = ws.get_project(PROJECT_ID)
    print(f"Reconnected to Project: {project.name}")
except Exception as e:
    print(f"Failed to find project: {e}")

# SILENCE WARNINGS
warnings.filterwarnings("ignore", message="Only one class is present in y_true")
warnings.filterwarnings("ignore", category=UserWarning)

# EXTRACT AND CLEAN
df_curr = eval_data_1._data.copy()
df_ref = eval_data_2._data.copy()

# Drop columns that are entirely NaN in EITHER dataset
curr_empty = df_curr.columns[df_curr.isnull().all()].tolist()
ref_empty = df_ref.columns[df_ref.isnull().all()].tolist()
all_empty = list(set(curr_empty + ref_empty))

# Drop columns with only ONE unique value 
constant_cols = [col for col in df_ref.columns if df_ref[col].nunique() <= 1]

# Drop specific known noise columns
ids_to_drop = ['id', 'member_id', 'url', 'desc']

total_drop = list(set(all_empty + constant_cols + ids_to_drop))

print(f"Scrubbing {len(total_drop)} problematic columns...")
df_curr.drop(columns=[c for c in total_drop if c in df_curr.columns], inplace=True)
df_ref.drop(columns=[c for c in total_drop if c in df_ref.columns], inplace=True)

# DATA MAPPING
target_col = 'is_default' if 'is_default' in df_curr.columns else None
pred_col = 'prediction' if 'prediction' in df_curr.columns else None

# Ensure targets are aligned
for df in [df_curr, df_ref]:
    if target_col: df[target_col] = df[target_col].astype(int)
    if pred_col: df[pred_col] = df[pred_col].astype(int)

all_cols = df_curr.columns.tolist()
features = [c for c in all_cols if c not in [target_col, pred_col]]
num_cols = df_curr[features].select_dtypes(include=['number']).columns.tolist()
cat_cols = df_curr[features].select_dtypes(exclude=['number']).columns.tolist()

data_def = DataDefinition(
    numerical_columns=num_cols,
    categorical_columns=cat_cols,
    classification=[
        BinaryClassification(target=target_col, prediction_labels=pred_col)
    ] if target_col and pred_col else []
)

# RUN AND UPLOAD
current_dataset = Dataset.from_pandas(df_curr, data_definition=data_def)
reference_dataset = Dataset.from_pandas(df_ref, data_definition=data_def)

# Class consistency check
curr_classes = set(df_curr[target_col].unique()) if target_col else set()
ref_classes = set(df_ref[target_col].unique()) if target_col else set()

metrics_list = [DataDriftPreset(), DataSummaryPreset()]
if target_col and pred_col and (curr_classes == ref_classes):
    metrics_list.append(ClassificationPreset())

report = Report(metrics=metrics_list)
print("Running the Final Stress-Test Analysis...")
my_eval = report.run(current_dataset, reference_dataset)

print(f"Uploading telemetry to Project ID: {project.id}...")
ws.add_run(project.id, my_eval, include_data=False)

print("-" * 30)
print(f"2026 Nigeria Stress-Test is live.")

Enter your Evidently Cloud API Token: ········
Reconnected to Project: MASTS-Stress-Testing-Monitoring
Scrubbing 27 problematic columns...
Running the Final Stress-Test Analysis...
Uploading telemetry to Project ID: 019d2e74-a7e4-7383-84ac-15385631afc4...
------------------------------
2026 Nigeria Stress-Test is live.


###### Dashboard

In [41]:
# Clear previous attempts
project.dashboard.panels = []

# Add the Overall Drift Panel 
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Nigeria Stress-Test: Overall Drift",
        size="half",
        values=[
            PanelMetric(
                legend="Share of Drifted Columns",
                metric="DatasetDriftMetric",
                field_path="share_of_drifted_columns" 
            ),
        ],
        plot_params={"plot_type": "line"},
    )
)

# Add the Feature Drift Panel 
project.dashboard.add_panel(
    DashboardPanelPlot(
        title="Macro Stress Index Drift Score",
        size="half",
        values=[
            PanelMetric(
                legend="p-value",
                metric="ColumnDriftMetric",
                metric_args={"column_name": "macro_stress_index"}, 
                field_path="drift_score"
            ),
        ],
        plot_params={"plot_type": "bar"},
    )
)

# Save to Cloud
project.save()
print(f"Dashboard logic synced for Project: {project.id}")

Dashboard logic synced for Project: 019d2e74-a7e4-7383-84ac-15385631afc4
